In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
#connecting to openrouter
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [ ]:
reader = PdfReader("me_ayesha/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
#adding my resume 
resume_reader = PdfReader("me_ayesha/Ayesha_resume.pdf")
resume = ""
for page in resume_reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [ ]:
print(resume)

In [ ]:
print(linkedin)

In [ ]:
with open("me_ayesha/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
# adding my portfolio website to gather information about my background and skills

def get_my_portfolio(url):
    res = requests.get(url, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    lines = soup.get_text(separator="\n", strip=True)
    return "\n".join(lines)

portfolio = get_my_portfolio("https://meayesha.github.io/portfolio/")
print(portfolio)

In [ ]:
name = "Ayesha Parveen"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background, LinkedIn profile, resume and portfolio which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n## Resume:\n{resume}\n\n## Portfolio:\n{portfolio}"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
system_prompt

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch()